# SELD 数据清单复核
只读取仓库中的元数据快照，不访问或修改原始数据。扫描实现见 `scripts/data/collect_inventory.py`；标签覆盖实现见 `scripts/data/check_label_coverage.py`。文件名/大小 digest 不是全内容 checksum。

In [ ]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == 'notebooks': root = root.parent
base = root / 'docs/data_inventory/2026-09-04'
snapshots = {host: json.loads((base / (host + '.json')).read_text(encoding='utf-8')) for host in ('rabbit02', 'RB05')}
[(host, x['asset'], x.get('files', 0), round(x.get('bytes', 0) / 2**30, 3), len(x['broken_symlinks'])) for host, d in snapshots.items() for x in d['assets']]

In [ ]:
coverage = json.loads((base / 'label_coverage.json').read_text(encoding='utf-8'))
[(x['host'], x['id'], x['audio'], x['labels'], x['matched'], len(x['sidecars'])) for x in coverage]

In [ ]:
migration = json.loads((base / 'migration_comparison.json').read_text(encoding='utf-8'))
assets = {host: {a['asset']: a for a in d['assets']} for host, d in snapshots.items()}
for row in migration:
    a, b = (assets[host][row['asset']] for host in ('rabbit02', 'RB05'))
    assert a['files'] == b['files'] == row['files']
    assert a['bytes'] == b['bytes'] == row['bytes']
    assert a['file_size_digest'] == b['file_size_digest']
    samples = lambda x: {(g['directory'], s['file']): s['sha256'] for g in x['groups'] for s in g['content_samples']}
    assert samples(a) == samples(b) and len(samples(a)) == row['content_sample_checks']
    full = lambda x: {g['directory']: (g['full_content_digest'], g['full_content_files']) for g in x['groups'] if 'full_content_digest' in g}
    assert full(a) == full(b)
    assert sum(count for _, count in full(a).values()) == row['full_content_checked_files']
assert sum(r['files'] for r in migration) == 6608
assert sum(r['content_sample_checks'] for r in migration) == 47
assert sum(r['full_content_checked_files'] for r in migration) == 802
migration